In [ ]:
# post_isin_country_expand.py
import pandas as pd
import re
from pathlib import Path

# ========= Adjust these paths =========
INPUT_CSV  = r"D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned_final.csv"
OUTPUT_CSV = r"D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned_ListedCountry-update.csv"
ISO_LOOKUP = r"D:\LinhDao\Programming\SUPERFUNdProject\iso_country_codes_complete.csv"  # expects Alpha-2 + Country (Friendly)
# =====================================

# Rare/legacy overrides (add more as needed)
OVERRIDES = {
    "AN": "United States",  # Netherlands Antilles (retired) -> USA (e.g., Schlumberger legacy ISIN)
}

ISIN_REGEX = re.compile(r"^[A-Z]{2}[A-Z0-9]{9}\d$")

def is_isin(x) -> bool:
    if pd.isna(x):
        return False
    s = str(x).strip().upper()
    return bool(ISIN_REGEX.fullmatch(s))

def load_iso_map(path: str) -> dict:
    """
    Load ISO map like {'US': 'United States', ...} using:
      - code column: 'Alpha-2'
      - country column: 'Country (Friendly)'
    Reads with utf-8-sig to strip BOM if present.
    """
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception as e:
        raise RuntimeError(f"Failed to read ISO lookup: {e}")

    col_code = "Alpha-2"
    col_country = "Country (Friendly)"
    if col_code not in df.columns or col_country not in df.columns:
        raise RuntimeError(
            f"Expected columns '{col_code}' and '{col_country}' in {path}. "
            f"Found: {list(df.columns)}"
        )

    mapping = {}
    for _, row in df.iterrows():
        code = str(row[col_code]).strip().upper()
        country = str(row[col_country]).strip()
        if code and country:
            mapping[code] = country
    return mapping

def main():
    # Load data (your cleaned file)
    df = pd.read_csv(INPUT_CSV, encoding="cp1252")

    # Guard: required columns
    required_cols = ["Stock ID", "Listed Country"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns: {missing}")

    # Load ISO code -> country mapping
    code_to_country = load_iso_map(ISO_LOOKUP)
    known_codes = set(code_to_country.keys()) | set(OVERRIDES.keys())

    # --- build masks safely (avoid creating "nan" strings) ---
    listed_raw = df["Listed Country"]
    listed = listed_raw.fillna("").astype(str).str.strip()  # blanks stay "", not "nan"

    stockid_str = df["Stock ID"].fillna("").astype(str).str.strip().str.upper()
    isin_mask = stockid_str.str.match(ISIN_REGEX)

    two_letter_mask = listed.str.match(r"^[A-Za-z]{2}$", na=False)
    candidates_mask = isin_mask & two_letter_mask

    before_norm = listed.copy()

    # Helper to map codes safely
    def code_to_friendly(code: str) -> str:
        u = str(code).strip().upper()
        if not u:
            return ""  # keep blanks blank
        if u in OVERRIDES:
            return OVERRIDES[u]
        return code_to_country.get(u, u)  # fall back to original code if unknown

    # Apply mapping ONLY on candidates; do not touch Stock ID
    mapped = listed.copy()
    mapped.loc[candidates_mask] = mapped[candidates_mask].apply(code_to_friendly)

    # Final clean-up: remove NaNs and any literal "nan"/"NaN"/"NAN" strings
    df["Listed Country"] = (
        mapped
        .replace(["nan", "NaN", "NAN"], "")
        .fillna("")
    )

    # Diagnostics (compare normalized before vs after)
    updated_rows_mask = candidates_mask & (df["Listed Country"] != before_norm)
    total_isin = int(isin_mask.sum())
    updated_count = int(updated_rows_mask.sum())
    not_updated_count = total_isin - updated_count

    # Distinct codes successfully mapped (from BEFORE values where change happened)
    mapped_codes = (
        before_norm[updated_rows_mask]
        .str.upper()
        .dropna()
        .unique()
        .tolist()
    )

    # Overrides applied count
    overrides_applied_count = int(
        (candidates_mask & listed.str.upper().isin(OVERRIDES.keys())).sum()
    )

    # --- Unmapped codes report (among ISIN candidates) ---
    cand_codes_upper = listed[candidates_mask].str.upper()
    unmapped_mask = candidates_mask & ~cand_codes_upper.isin(known_codes)
    unmapped_codes = (
        listed[unmapped_mask].str.upper().dropna().value_counts().sort_index()
    )

    # Write outputs
    df.to_csv(OUTPUT_CSV, index=False, encoding="cp1252", date_format="%b %d %Y")

    updated_rows_path = Path(OUTPUT_CSV).with_name(Path(OUTPUT_CSV).stem + "_ISIN_country_updates.csv")
    if updated_count > 0:
        df.loc[updated_rows_mask].to_csv(
            updated_rows_path, index=False, encoding="cp1252", date_format="%b %d %Y"
        )

    unmapped_rows_path = Path(OUTPUT_CSV).with_name(Path(OUTPUT_CSV).stem + "_ISIN_country_unmapped.csv")
    if int(unmapped_mask.sum()) > 0:
        df.loc[unmapped_mask].to_csv(
            unmapped_rows_path, index=False, encoding="cp1252", date_format="%b %d %Y"
        )

    # Summary
    print(f"Source file: {INPUT_CSV}")
    print(f"Full updated file written to: {OUTPUT_CSV}")
    if updated_count > 0:
        print(f"Updated rows file written to: {updated_rows_path}")
    else:
        print("No rows changed; updated-rows file not created.")
    print(f"ISIN-like rows detected: {total_isin}")
    print(f"✅ ISIN rows updated: {updated_count}")
    print(f"⏭️ ISIN rows not updated: {not_updated_count}")
    if mapped_codes:
        print(f"Distinct 2-letter codes mapped → friendly names: {sorted(mapped_codes)}")
    if overrides_applied_count > 0:
        print(f"Overrides applied (e.g., AN→United States): {overrides_applied_count}")
    if int(unmapped_mask.sum()) > 0:
        print(f"⚠️ Unmapped ISIN candidate rows: {int(unmapped_mask.sum())}")
        print(f"Unmapped codes and counts:\n{unmapped_codes.to_string()}")
        print(f"Unmapped rows file written to: {unmapped_rows_path}")
    else:
        print("No unmapped ISIN candidate codes 🎉")

if __name__ == "__main__":
    main()


Source file: D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned_final.csv
Full updated file written to: D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned_ListedCountry-update.csv
Updated rows file written to: D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned_ListedCountry-update_ISIN_country_updates.csv
ISIN-like rows detected: 3252
✅ ISIN rows updated: 3252
⏭️ ISIN rows not updated: 0
Distinct 2-letter codes mapped → friendly names: ['AE', 'AN', 'AT', 'AU', 'BE', 'BM', 'BR', 'CA', 'CH', 'CN', 'DE', 'DK', 'ES', 'FI', 'FR', 'GB', 'GG', 'GR', 'HK', 'HU', 'ID', 'IE', 'IL', 'IM', 'IN', 'IT', 'JE', 'JP', 'KR', 'KY', 'LR', 'LU', 'MU', 'MX', 'MY', 'NL', 'NO', 'NZ', 'PA', 'PH', 'PL', 'PR', 'PT', 'QA', 'SE', 'SG', 'TH', 'TR', 'TW', 'US', 'VG', 'ZA']
Overrides applied (e.g., AN→United States): 1
No unmapped ISIN candidate codes 🎉


In [4]:
# Debug: show raw headers as parsed by pandas
ISO_LOOKUP = r"D:\LinhDao\Programming\SUPERFUNdProject\iso_country_codes_complete.csv"  # expects country + 2-letter code
tmp = pd.read_csv(ISO_LOOKUP, encoding="utf-8-sig", nrows=0)
print("Headers in ISO_LOOKUP:", list(tmp.columns))

Headers in ISO_LOOKUP: ['Alpha-2', 'Alpha-3', 'Numeric', 'Country (Friendly)', 'Country (Official)']
